# 0 · Deploy a Custom Chat App for the Supervisor Agent

Deploys a **Custom Chat App** on **Databricks Apps** — a production Streamlit interface pointed at the
Supervisor Agent's model serving endpoint created in notebook `3`.

The app source lives as plain files in the **`chat_app/`** folder next to this
notebook (`app.py`, `app.yaml`, `requirements.txt`). This notebook just binds
the serving endpoint as an app resource and deploys that folder — nothing to
edit by hand.

**What this notebook does**
1. Locates the sibling `chat_app/` source folder in the workspace
2. Creates the App with the Supervisor serving endpoint bound as a resource
   (`CAN_QUERY`) under the key `agent-endpoint` — the app's service principal
   gets query access automatically, and the endpoint name is injected into the
   app via `app.yaml`'s `valueFrom: agent-endpoint` (no hardcoding)
3. Deploys the app and prints its URL

### Install Dependencies

In [0]:
%pip install --upgrade databricks-sdk
dbutils.library.restartPython()

### Define Parameters
Review the widgets before running. Defaults match the **Marketing Campaign Analytics** layout.
| Widget | Default | What it controls |
| --- | --- | --- |
| `CATALOG_NAME` | `workspace` | Unity Catalog for demo objects. Default `workspace` is pre-provisioned — no catalog creation needed. |
| `SCHEMA_NAME` | `marketing_campaign` | Schema inside the catalog for sales data and contracts. |
| `APP_NAME` | `marketing-app` | Databricks App name (lowercase, hyphens). |
| `SUPERVISOR_NAME` | `Marketing Campaign Insights Supervisor` | Used to resolve the serving endpoint when `SERVING_ENDPOINT_NAME` is blank. |
| `SERVING_ENDPOINT_NAME` | *(empty)* | Supervisor model-serving endpoint. Leave blank to resolve from `SUPERVISOR_NAME`. |
| `APP_SOURCE_PATH` | `./chat_app` | Folder with `app.py`, `app.yaml`, and `requirements.txt`. |


In [0]:
dbutils.widgets.text("APP_NAME", "marketing-app", "APP_NAME — Databricks App name (lowercase, hyphens)")
dbutils.widgets.text("SUPERVISOR_NAME", "Marketing Campaign Insights Supervisor", "SUPERVISOR_NAME — display name from notebook 3")
dbutils.widgets.text("SERVING_ENDPOINT_NAME", "", "SERVING_ENDPOINT_NAME — leave blank to resolve from SUPERVISOR_NAME")
dbutils.widgets.text("CATALOG_NAME", "workspace", "CATALOG_NAME — catalog with tables the Genie space queries")
dbutils.widgets.text("SCHEMA_NAME", "marketing_campaign", "SCHEMA_NAME — schema with sales tables and metric view")
dbutils.widgets.text("APP_SOURCE_PATH", "./chat_app", "App source folder (blank = ./chat_app next to this notebook)")

APP_NAME = dbutils.widgets.get("APP_NAME")
SUPERVISOR_NAME = dbutils.widgets.get("SUPERVISOR_NAME")
SERVING_ENDPOINT_NAME = dbutils.widgets.get("SERVING_ENDPOINT_NAME")
CATALOG_NAME = dbutils.widgets.get("CATALOG_NAME")
SCHEMA_NAME = dbutils.widgets.get("SCHEMA_NAME")
APP_SOURCE_PATH = dbutils.widgets.get("APP_SOURCE_PATH")

# The app resource key. MUST match `valueFrom: agent-endpoint` in chat_app/app.yaml.
RESOURCE_KEY = "agent-endpoint"

### Initialize Client and Resolve Paths

In [0]:
import os
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

# Resolve the endpoint from the supervisor by name if not provided explicitly.
if not SERVING_ENDPOINT_NAME:
    sup = next((s for s in w.supervisor_agents.list_supervisor_agents()
                if s.display_name == SUPERVISOR_NAME), None)
    assert sup is not None, f"Supervisor '{SUPERVISOR_NAME}' not found. Run notebook 3 first."
    SERVING_ENDPOINT_NAME = sup.endpoint_name

# Resolve the app source folder to an absolute workspace path.
# The deploy API requires an absolute /Workspace/... path, so a relative value
# like "./chat_app" is resolved against this notebook's directory.
ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
nb_dir = os.path.dirname(ctx.notebookPath().get())  # e.g. /Repos/<user>/<repo>/4_apps
if not APP_SOURCE_PATH:
    APP_SOURCE_PATH = f"/Workspace{nb_dir}/chat_app"
elif not APP_SOURCE_PATH.startswith("/"):
    rel = APP_SOURCE_PATH.lstrip("./").strip("/")
    APP_SOURCE_PATH = f"/Workspace{nb_dir}/{rel}"
# else: an absolute /Workspace/... (or /Repos/...) path is used as-is

print("Endpoint        :", SERVING_ENDPOINT_NAME)
print("App source path :", APP_SOURCE_PATH)

# Sanity-check the source files are present so the deploy can't silently ship an empty app.
required = ["app.py", "app.yaml", "requirements.txt"]
present = os.listdir(APP_SOURCE_PATH)
missing = [f for f in required if f not in present]
assert not missing, f"Missing {missing} in {APP_SOURCE_PATH}. Import the chat_app/ folder into the workspace."
print("Source files    :", present)

### Create the App (bind the serving endpoint as a resource)

In [0]:
from databricks.sdk.service.apps import (
    App, AppResource, AppResourceServingEndpoint,
    AppResourceServingEndpointServingEndpointPermission as EndpointPermission,
)

endpoint_resource = AppResource(
    name=RESOURCE_KEY,
    serving_endpoint=AppResourceServingEndpoint(
        name=SERVING_ENDPOINT_NAME,
        permission=EndpointPermission.CAN_QUERY,
    ),
)
app_spec = App(name=APP_NAME, description=f"Custom chat app for {SUPERVISOR_NAME}",
               resources=[endpoint_resource])

existing = next((a for a in w.apps.list() if a.name == APP_NAME), None)
if existing:
    print(f"App '{APP_NAME}' already exists; updating resources.")
    app = w.apps.update(name=APP_NAME, app=app_spec)
else:
    print(f"Creating app '{APP_NAME}' (provisioning compute, ~2-4 min)...")
    app = w.apps.create_and_wait(app=app_spec)
print("App service principal:", app.service_principal_name)
print("App status:", app.app_status.state if app.app_status else None)

### Grant the App's Service Principal Access to Underlying Resources

The app calls the Supervisor endpoint, and the Supervisor — running as the
**app's service principal** — in turn calls its sub-agents (the Vector Search
index and the Genie space) and reads the Genie's Unity Catalog tables via a SQL
warehouse. The app SP therefore needs access to *all* of these. This cell
resolves the Supervisor's tools automatically and grants:

- **Embedding endpoint** (`databricks-gte-large-en`, used for query-time
  embeddings by the Vector Search index) → `CAN_QUERY`
- **Genie space(s)** → `CAN_RUN`
- **SQL warehouse** backing each Genie space → `CAN_USE`
- **Unity Catalog** `USE CATALOG` + schema `USE SCHEMA` + `SELECT` — covers all
  tables, metric views, **and the Vector Search index** in the schema

In [0]:
from databricks.sdk.service.serving import (
    ServingEndpointAccessControlRequest, ServingEndpointPermissionLevel)
from databricks.sdk.service.sql import (
    WarehouseAccessControlRequest, WarehousePermissionLevel)
from databricks.sdk.service.iam import AccessControlRequest, PermissionLevel
from databricks.sdk.service.catalog import PermissionsChange, SecurableType, Privilege

# Application ID of the app's service principal — the identity the supervisor
# uses when it fans out to the sub-agents.
SP = app.service_principal_client_id
print("Granting resource access to app service principal:", SP)

# Resolve the supervisor and enumerate its tools (Vector Search index + Genie).
sup = next((s for s in w.supervisor_agents.list_supervisor_agents()
            if s.endpoint_name == SERVING_ENDPOINT_NAME or s.display_name == SUPERVISOR_NAME), None)
assert sup is not None, "Could not resolve the supervisor agent."
tools = w.api_client.do("GET", f"/api/2.1/{sup.name}/tools").get("tools", [])

def _field(tool, key, *names):
    d = tool.get(key) or {}
    for n in names:
        if d.get(n):
            return d[n]
    return None

genie_space_ids = [_field(t, "genie_space", "id", "space_id")
                   for t in tools if t["tool_type"] == "genie_space"]
vs_indexes = [_field(t, "vector_search_index", "name")
              for t in tools if t["tool_type"] == "vector_search_index"]
print("Genie spaces:", genie_space_ids, "| Vector Search indexes:", vs_indexes)

# 1) Embedding endpoint (query-time embeddings for the VS index) -> CAN_QUERY
for emb in ["databricks-gte-large-en"]:
    try:
        eid = w.serving_endpoints.get(emb).id
        w.serving_endpoints.update_permissions(
            serving_endpoint_id=eid,
            access_control_list=[ServingEndpointAccessControlRequest(
                service_principal_name=SP,
                permission_level=ServingEndpointPermissionLevel.CAN_QUERY)])
        print("  CAN_QUERY on embedding endpoint", emb)
    except Exception as e:
        print("  (warn) embedding endpoint grant:", str(e)[:140])

# 1b) Vector Search index -> SELECT on the index + CAN_USE on its serving endpoint
for idx_name in vs_indexes:
    try:
        w.grants.update(securable_type="table", full_name=idx_name,
                        changes=[PermissionsChange(principal=SP, add=[Privilege.SELECT])])
        vidx = w.vector_search_indexes.get_index(index_name=idx_name)
        vep = w.vector_search_endpoints.get_endpoint(vidx.endpoint_name)
        w.api_client.do("PATCH", f"/api/2.0/permissions/vector-search-endpoints/{vep.id}",
                        body={"access_control_list": [{"service_principal_name": SP,
                                                       "permission_level": "CAN_USE"}]},
                        headers={"Accept": "application/json", "Content-Type": "application/json"})
        print("  SELECT on index + CAN_USE on VS endpoint for", idx_name)
    except Exception as e:
        print("  (warn) vector search grant:", str(e)[:140])

# 2) Genie spaces -> CAN_RUN, and collect their backing warehouses
warehouse_ids = set()
for gid in genie_space_ids:
    w.permissions.update(
        request_object_type="genie", request_object_id=gid,
        access_control_list=[AccessControlRequest(
            service_principal_name=SP, permission_level=PermissionLevel.CAN_RUN)])
    print("  CAN_RUN on genie space", gid)
    try:
        space = w.api_client.do("GET", f"/api/2.0/genie/spaces/{gid}")
        if space.get("warehouse_id"):
            warehouse_ids.add(space["warehouse_id"])
    except Exception as e:
        print("  (warn) could not read warehouse for genie space", gid, ":", e)

# 3) SQL warehouses -> CAN_USE
for wid in warehouse_ids:
    w.warehouses.update_permissions(
        warehouse_id=wid,
        access_control_list=[WarehouseAccessControlRequest(
            service_principal_name=SP, permission_level=WarehousePermissionLevel.CAN_USE)])
    print("  CAN_USE on warehouse", wid)

# 4) Unity Catalog read access — covers all tables, metric views, AND the
#    Vector Search index (all live in {CATALOG}.{SCHEMA}).
w.grants.update(securable_type=SecurableType.CATALOG.value, full_name=CATALOG_NAME,
                changes=[PermissionsChange(principal=SP, add=[Privilege.USE_CATALOG])])
w.grants.update(securable_type=SecurableType.SCHEMA.value,
                full_name=f"{CATALOG_NAME}.{SCHEMA_NAME}",
                changes=[PermissionsChange(principal=SP,
                                           add=[Privilege.USE_SCHEMA, Privilege.SELECT])])
print(f"  USE CATALOG on {CATALOG_NAME}; USE SCHEMA + SELECT on {CATALOG_NAME}.{SCHEMA_NAME}")
print("\nAll grants applied. (Permissions can take ~1 min to propagate.)")

### Deploy the App

In [0]:
from databricks.sdk.service.apps import AppDeployment

print("Deploying app source from", APP_SOURCE_PATH, "...")
deployment = w.apps.deploy_and_wait(
    app_name=APP_NAME,
    app_deployment=AppDeployment(source_code_path=APP_SOURCE_PATH),
)
print("Deployment state:", deployment.status.state if deployment.status else None)
if deployment.status and deployment.status.message:
    print("Message:", deployment.status.message)

### Result

In [0]:
app = w.apps.get(name=APP_NAME)
print("App URL   :", app.url)
print("App logs  :", (app.url.rstrip('/') + "/logz") if app.url else None)
print("Endpoint  :", SERVING_ENDPOINT_NAME)
print("\nOpen the App URL to chat with the Supervisor Agent.")